<a href="https://colab.research.google.com/github/KseniiaKriuchkova/Information-Retrieval-Project/blob/main/IR_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install rank-bm25
import kagglehub
import pandas as pd
import re
import os
import spacy
import numpy as np
from rank_bm25 import BM25Okapi


In [ ]:
path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")

print("Path to dataset files:", path)

In [ ]:
print("Files in dataset directory:")
for f in os.listdir(path):
    print(f)

In [ ]:
data = pd.read_csv("/kaggle/input/ag-news-classification-dataset/test.csv")
data.head(50)

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower().strip()
    text = re.sub(r'#\d+', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)

    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]

    return " ".join(tokens)


In [ ]:
data_sample = data.copy()
data_sample['clean_text'] = (data_sample['Title'].fillna('') + " " +
                             data_sample['Description'].fillna('')).apply(preprocess_text)

In [ ]:
tokenized_corpus = [doc.split() for doc in data_sample['clean_text']]
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
queries = [
    "Methods for detecting cancer symptoms",
    "Next space mission after Columbia",
    "Apple launches new product",
    "Who are gold medal winners in Athens?",
    "UN mediating Middle East conflicts",
    "Urgent climate issues worldwide",
    "Airplane accident in Latin America",
    "New technology against spam",
    "dictator court case",
    "Terroristic act in Russia"

]
for i, query in  enumerate(queries, 1):
    print(f"\nResults for query {i}: {query}")
    print("=" * 80)

    query_tokens = preprocess_text(query).split()
    scores = bm25.get_scores(query_tokens)
    top_n = 5
    top_n_idx = np.argsort(scores)[::-1][:top_n]

    for idx in top_n_idx:
      print(f"Score: {scores[idx]:.2f}")
      print(f"Title: {data_sample['Title'].iloc[idx]}")
      print(f"Description: {data_sample['Description'].iloc[idx]}")
      print("-" * 80)

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
doc_embeddings = model.encode(
    data_sample['clean_text'].tolist(),
    convert_to_tensor=True
)

In [ ]:
query_embeddings = model.encode(
    queries,
    convert_to_tensor=True
)

In [ ]:
similarity_scores = util.cos_sim(query_embeddings, doc_embeddings)

In [ ]:
top_k = 5
for q_idx, query in enumerate(queries):

    scores = similarity_scores[q_idx]

    top_results = torch.topk(scores, k=top_k)

    print(f"\nQuery: {query}")
    print("=" * 60)

    for score, idx in zip(top_results.values, top_results.indices):

        idx = idx.item()
        score = score.item()

        print(f"Score: {score:.4f}")
        print("Title:", data_sample['Title'].iloc[idx])
        print("Description:", data_sample['Description'].iloc[idx])
        print("-" * 60)

In [ ]:
!pip install faiss-cpu
import faiss

In [ ]:
doc_embeddings = model.encode(data_sample['clean_text'].tolist(), convert_to_numpy=True)

In [ ]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

In [ ]:
query_embedding = model.encode(queries, convert_to_numpy=True)
top_k = 5
distances, indices = index.search(query_embedding, top_k)

In [ ]:
for q_idx, query in enumerate(queries):
    print(f"Query: {query}")
    print("=" * 60)

    for rank, doc_idx in enumerate(indices[q_idx], start=1):
        print(f"Rank: {rank}")
        print(f"Distance: {distances[q_idx][rank-1]:.2f}")
        print("Title:", data_sample['Title'].iloc[doc_idx])
        print("Description:", data_sample['Description'].iloc[doc_idx])
        print("-" * 60)

    print("\n")
